# CNN + MAE Pre-training for Ultrasound ViT

Trains a **CNN + Masked Autoencoder (MAE)** Vision Transformer.  A configurable
multi-layer CNN extracts feature maps from raw images before ViT tokenisation.

## Key concept: `pixel_patch_size`
Each ViT token covers a region of `patch_size × cnn_effective_stride` pixels on
each side of the *original* image.  Images must be padded to multiples of this
value.  Example: `patch_size=16`, strides `[1, 2]` → `effective_stride=2`,
`pixel_patch_size=32`.

## Workflow
1. Load dataset
2. Configure CNN + model & training hyperparameters
3. **Train** with `train_mae` from `embeddings.cnn_vit.train`
4. Visualise reconstructions
5. Extract embeddings / linear probe

In [4]:
!ls /content

datasets_adapters  drive  embeddings  sample_data


In [3]:
%matplotlib inline

import math, os, sys
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output

print(Path.cwd())
PROJECT_ROOT = str(Path.cwd().resolve().parents[0])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from embeddings.cnn_vit.train import (
    MAETrainTransform,
    MAEValTransform,
    train_mae,
    load_checkpoint,
    visualize_reconstruction,
    dump_mae_training_metrics_artifacts,
)

print(f'Project root: {PROJECT_ROOT}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

/content
Project root: /
PyTorch: 2.10.0+cu128
CUDA available: True


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Load Dataset

In [6]:
import shutil
from datasets_adapters.fetal_planes_db.fpd_dataset import FetalPlanesDBDataset
from datasets_adapters.fetal_head_circ.fhc_dataset import FetalHeadCircDataset
from datasets_adapters.focus import FOCUSDataset
from datasets_adapters.abdominal_segmentation import AbdominalSegmentationDataset
from torch.utils.data import ConcatDataset

DATASET_ROOT  = '/content/drive/MyDrive/SHAD/project/fetal_planes_db'
FHC_IMAGES_DIR = '/content/drive/MyDrive/SHAD/project/fetal_head_circumference/training_set/training_set'
FHC_CSV        = '/content/drive/MyDrive/SHAD/project/fetal_head_circumference/training_set_pixel_size_and_HC.csv'
FOCUS_ROOT     = '/content/drive/MyDrive/SHAD/project/focus'
ABDOMINAL_ROOT = '/content/drive/MyDrive/SHAD/project/FetalAbdominalSegmentationDataset'



dataset = ConcatDataset([
    FetalPlanesDBDataset(root=DATASET_ROOT, images_dir='Images'),
    FetalHeadCircDataset(images_dir=FHC_IMAGES_DIR, csv_file=FHC_CSV, load_annotations=False),
    FOCUSDataset(root=FOCUS_ROOT, split='training', load_masks=False),
    AbdominalSegmentationDataset(root=ABDOMINAL_ROOT, load_masks=False),
])
print(f'Dataset size: {len(dataset)} images')

Loaded 12400 images from /content/drive/MyDrive/SHAD/project/fetal_planes_db
Loaded 999 images from /content/drive/MyDrive/SHAD/project/fetal_head_circumference/training_set/training_set
Loaded 200 images from /content/drive/MyDrive/SHAD/project/focus/training
Loaded 1588 samples from /content/drive/MyDrive/SHAD/project/FetalAbdominalSegmentationDataset/ARRAY_FORMAT
Dataset size: 15187 images


## 2. Configuration

### CNN Architecture

| Variable | Meaning |
|---|---|
| `CNN_LAYER_CHANNELS` | Output channels per conv layer. Length = number of layers. |
| `CNN_KERNEL_SIZES` | Kernel size(s). Int = same for all layers; list = per layer. |
| `CNN_STRIDES` | Stride(s). Product of strides = `effective_stride`. |

**`pixel_patch_size = PATCH_SIZE × effective_stride`** — the region of the
original image each ViT token covers.  `MAX_IMAGE_HEIGHT` must be divisible
by `pixel_patch_size`.

**Examples:**
```python
# No spatial reduction (feature extraction only)
CNN_LAYER_CHANNELS = [32, 64]  ;  CNN_STRIDES = 1       # pixel_patch_size = 16

# 2× downsampling before patching
CNN_LAYER_CHANNELS = [32, 64]  ;  CNN_STRIDES = [1, 2]  # pixel_patch_size = 32

# 4× downsampling with larger receptive field
CNN_LAYER_CHANNELS = [32, 64, 128]  ;  CNN_STRIDES = [1, 2, 2]  # pixel_patch_size = 64
```

In [ ]:
# ── CNN Feature Extractor ───────────────────────────────────────────
CNN_LAYER_CHANNELS = [32, 64]   # one int per layer
CNN_KERNEL_SIZES   = 3          # int or list[int] — kernel size per layer
CNN_STRIDES        = 1          # int or list[int] — stride per layer
                                # effective_stride = product(CNN_STRIDES)
                                # pixel_patch_size = PATCH_SIZE * effective_stride

# ── ViT Model ──────────────────────────────────────────────────────
MAX_IMAGE_HEIGHT      = 224     # must be divisible by pixel_patch_size
PATCH_SIZE            = 16
EMBED_DIM             = 512
DEPTH                 = 8
NUM_HEADS             = 8
DECODER_EMBED_DIM     = 384
DECODER_DEPTH         = 6
DECODER_NUM_HEADS     = 8
DECODER_PRED_NUM_LAYERS = 3
MLP_RATIO             = 4.0
NORM_PIX_LOSS         = True
CLIP_PIXEL_PRED       = False
L1_RATIO              = 0.9
FFT_LOSS_WEIGHT       = 0.1

# ── Training ──────────────────────────────────────────────────────
EPOCHS        = 200
BATCH_SIZE    = 128
LR            = 1.5e-4
WEIGHT_DECAY  = 0.05
WARMUP_EPOCHS = 10
MIN_LR        = 1e-5
MASK_RATIO    = 0.75
VAL_SPLIT     = 0.1
NUM_WORKERS   = 8

# ── Checkpointing ─────────────────────────────────────────────────
CHECKPOINT_DIR  = os.path.join('/content/drive/MyDrive/SHAD/project', 'checkpoints', 'cnn_mae', 'v2.1')
SAVE_EVERY      = 20
VISUALIZE_EVERY = 10
VIS_INDICES     = [42, 137, 1000, 1100]

# ── Device ─────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Derived: pixel_patch_size ───────────────────────────────────────
_strides = [CNN_STRIDES] * len(CNN_LAYER_CHANNELS) if isinstance(CNN_STRIDES, int) else list(CNN_STRIDES)
_eff = 1
for s in _strides:
    _eff *= s
PIXEL_PATCH_SIZE = PATCH_SIZE * _eff
print(f'CNN effective_stride={_eff}, pixel_patch_size={PIXEL_PATCH_SIZE}')
assert MAX_IMAGE_HEIGHT % PIXEL_PATCH_SIZE == 0, (
    f'MAX_IMAGE_HEIGHT={MAX_IMAGE_HEIGHT} must be divisible by pixel_patch_size={PIXEL_PATCH_SIZE}'
)

Device: cuda
CNN effective_stride=1, pixel_patch_size=16


## 3. Train

In [8]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def update_plots(history: dict, epochs_total: int) -> None:
    clear_output(wait=True)
    epochs_so_far = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    ax = axes[0]
    ax.plot(epochs_so_far, history['train_loss'], label='train', linewidth=1.5)
    ax.plot(epochs_so_far, history['val_loss'],   label='val',   linewidth=1.5)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Reconstruction Loss'); ax.set_title('Loss')
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(epochs_so_far, history['train_loss'], label='train', linewidth=1.5)
    ax.plot(epochs_so_far, history['val_loss'],   label='val',   linewidth=1.5)
    ax.set_yscale('log')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (log)'); ax.set_title('Loss (log scale)')
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(epochs_so_far, history['lr'], color='tab:orange', linewidth=1.5)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Learning Rate'); ax.set_title('LR Schedule')
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
    ax.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()
    e = len(history['train_loss'])
    print(
        f"Epoch {e}/{epochs_total}  |  "
        f"train_loss={history['train_loss'][-1]:.4f}  |  "
        f"val_loss={history['val_loss'][-1]:.4f}  |  "
        f"lr={history['lr'][-1]:.2e}"
    )


def on_epoch_end(epoch, history, model):
    update_plots(history, EPOCHS)
    if (epoch + 1) % VISUALIZE_EVERY == 0:
        vis_path = os.path.join(CHECKPOINT_DIR, f'reconstruction_epoch_{epoch + 1}.png')
        visualize_reconstruction(
            model, dataset,
            num_samples=len(VIS_INDICES), mask_ratio=MASK_RATIO,
            device=DEVICE, save_path=vis_path, sample_indices=VIS_INDICES,
        )
        dump_mae_training_metrics_artifacts(history, CHECKPOINT_DIR, epoch + 1)

In [9]:
model, history = train_mae(
    dataset,
    # ── CNN ──────────────────────────────────────────────────────────
    cnn_layer_channels=CNN_LAYER_CHANNELS,
    cnn_kernel_sizes=CNN_KERNEL_SIZES,
    cnn_strides=CNN_STRIDES,
    # ── ViT ──────────────────────────────────────────────────────────
    max_image_height=MAX_IMAGE_HEIGHT,
    patch_size=PATCH_SIZE,
    embed_dim=EMBED_DIM,
    depth=DEPTH,
    num_heads=NUM_HEADS,
    decoder_embed_dim=DECODER_EMBED_DIM,
    decoder_depth=DECODER_DEPTH,
    decoder_num_heads=DECODER_NUM_HEADS,
    mlp_ratio=MLP_RATIO,
    clip_pixel_pred=CLIP_PIXEL_PRED,
    norm_pix_loss=NORM_PIX_LOSS,
    decoder_pred_num_layers=DECODER_PRED_NUM_LAYERS,
    l1_loss_weight=L1_RATIO,
    l2_loss_weight=1 - L1_RATIO,
    fft_loss_weight=FFT_LOSS_WEIGHT,
    # ── Training ─────────────────────────────────────────────────────
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    min_lr=MIN_LR,
    mask_ratio=MASK_RATIO,
    val_split=VAL_SPLIT,
    num_workers=NUM_WORKERS,
    checkpoint_dir=CHECKPOINT_DIR,
    save_every=SAVE_EVERY,
    device=DEVICE,
    use_amp=True,
    on_epoch_end=on_epoch_end,
    results_json=os.path.join(CHECKPOINT_DIR, 'embed_history.json'),
)

Epoch 39/200  |  train_loss=0.5454  |  val_loss=0.5476  |  lr=1.43e-04


: 

In [ ]:
from google.colab import runtime
print('Task completed. Terminating session...')
runtime.terminate()

## 4. Visualise Reconstructions

In [ ]:
visualize_reconstruction(
    model, dataset,
    num_samples=5,
    mask_ratio=MASK_RATIO,
    device=DEVICE,
    sample_indices=VIS_INDICES,
)

## 5. Resume Training (optional)

In [ ]:
RESUME_FROM = os.path.join(CHECKPOINT_DIR, 'mae_epoch_60.pt')

model, history = train_mae(
    dataset,
    cnn_layer_channels=CNN_LAYER_CHANNELS,
    cnn_kernel_sizes=CNN_KERNEL_SIZES,
    cnn_strides=CNN_STRIDES,
    max_image_height=MAX_IMAGE_HEIGHT,
    patch_size=PATCH_SIZE,
    embed_dim=EMBED_DIM,
    depth=DEPTH,
    num_heads=NUM_HEADS,
    decoder_embed_dim=DECODER_EMBED_DIM,
    decoder_depth=DECODER_DEPTH,
    decoder_num_heads=DECODER_NUM_HEADS,
    mlp_ratio=MLP_RATIO,
    clip_pixel_pred=CLIP_PIXEL_PRED,
    norm_pix_loss=NORM_PIX_LOSS,
    decoder_pred_num_layers=DECODER_PRED_NUM_LAYERS,
    l1_loss_weight=L1_RATIO,
    l2_loss_weight=1 - L1_RATIO,
    fft_loss_weight=FFT_LOSS_WEIGHT,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    min_lr=MIN_LR,
    mask_ratio=MASK_RATIO,
    val_split=VAL_SPLIT,
    num_workers=NUM_WORKERS,
    checkpoint_dir=CHECKPOINT_DIR,
    save_every=SAVE_EVERY,
    device=DEVICE,
    use_amp=True,
    on_epoch_end=on_epoch_end,
    results_json=os.path.join(CHECKPOINT_DIR, 'embed_history.json'),
    resume_from=RESUME_FROM,
)

## 6. Linear Probe Evaluation

In [ ]:
from datasets_adapters.fetal_planes_db.linear_probe import train_linear_probe

encoder, _ = load_checkpoint(os.path.join(CHECKPOINT_DIR, 'mae_final.pt'))
encoder.to(DEVICE)

probe_history = train_linear_probe(
    encoder=encoder,
    embed_dim=EMBED_DIM,
    dataset_root=DATASET_ROOT,
    images_dir='Images',
    max_image_height=MAX_IMAGE_HEIGHT,
    epochs=100,
    batch_size=64,
    lr=1e-3,
    weight_decay=1e-4,
    device=DEVICE,
    checkpoint_dir=os.path.join(CHECKPOINT_DIR, 'linear_probe'),
    save_every=10,
    num_workers=8,
    show_plot=True,
)

print(f"Best val accuracy: {100 * max(probe_history['val_acc']):.2f}%")